# 05C_CNN_Feature_Extraction.ipynb
Loads the trained ResNet18 encoder, removes the classification head, extracts feature vectors for every GeoTIFF image, and saves them as `cnn_features.csv`.

In [ ]:

!pip -q install rasterio torch torchvision pandas

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18

ROOT='/content/drive/MyDrive/FloodProject'
SENT=os.path.join(ROOT,'Sentinel')
LAND=os.path.join(ROOT,'Landsat')

MODEL_PATH='/content/cnn_encoder_resnet18.pth'   # Upload if not present

IMAGE_SIZE=224

# Upload model if needed
if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print("Upload cnn_encoder_resnet18.pth")
    up=files.upload()
    MODEL_PATH=list(up.keys())[0]

files_list=glob.glob(os.path.join(SENT,'*.tif'))
files_list+=glob.glob(os.path.join(LAND,'*.tif'))

model=resnet18(weights=None)

old=model.conv1
model.conv1=nn.Conv2d(
    4,
    old.out_channels,
    kernel_size=old.kernel_size,
    stride=old.stride,
    padding=old.padding,
    bias=False
)

# Temporary FC (matches previous notebook)
model.fc=nn.Linear(model.fc.in_features,3)

state=torch.load(MODEL_PATH,map_location='cpu')
model.load_state_dict(state)

# Remove classification layer
encoder=nn.Sequential(*list(model.children())[:-1])
encoder.eval()

rows=[]

for fp in files_list:

    with rasterio.open(fp) as src:
        img=src.read().astype(np.float32)

    img=np.clip(img/10000.0,0,1)

    x=torch.from_numpy(img)
    x=F.interpolate(
        x.unsqueeze(0),
        size=(IMAGE_SIZE,IMAGE_SIZE),
        mode='bilinear',
        align_corners=False
    )

    with torch.no_grad():
        feat=encoder(x).squeeze().numpy()

    row={"event_id":os.path.splitext(os.path.basename(fp))[0]}

    for i,v in enumerate(feat):
        row[f"feature_{i+1}"]=float(v)

    rows.append(row)

df=pd.DataFrame(rows)

OUT=os.path.join(ROOT,"cnn_features.csv")
df.to_csv(OUT,index=False)

print(df.head())
print("Saved:",OUT)
